In [1]:
import os
from dotenv import load_dotenv
from typing import Annotated, TypedDict
import operator

from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()


True

In [2]:
# ── State Definition ──────────────────────────────────────────────────────────
class AgentState(TypedDict):
    """
    Shared state across all nodes in the graph.

    messages: full conversation history
      - Annotated with operator.add means new messages are APPENDED
      - Without this, each node write would REPLACE the list
    iteration_count: how many model calls have been made
    max_iterations: safety limit — stop if exceeded
    """
    messages: Annotated[list[BaseMessage], operator.add]
    iteration_count: int
    max_iterations: int

In [3]:
# ── Tools ─────────────────────────────────────────────────────────────────────

@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression. Input: valid Python math expression."""
    try:
        import math
        result = eval(expression, {"__builtin__": {}}, {"math": math})
        return str(result)
    except Exception as e:
        return f"Error evaluating expression: {e}"
    

@tool
def get_backend_fact(topic: str) -> str:
    """
    Get a key fact about a backend engineering topic.
    Topics: redis, postgresql, docker, kubernetes, jwt, fastapi
    """
    facts = {
        "redis": "Redis stores data in memory — reads are O(1). Use TTL on all keys to prevent memory exhaustion.",
        "postgresql": "PostgreSQL uses MVCC for isolation. Each transaction sees a consistent snapshot from when it started.",
        "docker": "Docker containers share the host kernel but isolate processes via Linux namespaces and cgroups.",
        "kubernetes": "Kubernetes schedules pods across nodes. Services provide stable DNS names for pods that restart.",
        "jwt": "JWT = header.payload.signature. Access tokens: 15-60 min. Refresh tokens: 7-30 days, store in DB for revocation.",
        "fastapi": "FastAPI generates OpenAPI docs automatically from type hints. Async endpoints use asyncio for non-blocking I/O.",
    }
    return facts.get(topic.lower(), "Topic not found. Valid topics: redis, postgresql, docker, kubernetes, jwt, fastapi.")

TOOLS = [calculate, get_backend_fact]

TOOL_MAP = { t.name: t for t in TOOLS }


In [4]:
# ── LLM with tools bound ──────────────────────────────────────────────────────

llm = ChatOpenAI(model="gpt-5-nano", temperature=0)
llm_with_tools = llm.bind_tools(TOOLS)

In [5]:
# ── Nodes ─────────────────────────────────────────────────────────────────────

def model_node(state: AgentState) -> AgentState:
    """
    Call the LLM with current conversation history.
    The LLM decides whether to call a tool or give a final answer.
    """
    response = llm_with_tools.invoke(state["messages"])
    return {
        "messages": [response],
        "iteration_count": state["iteration_count"] + 1,
    }

def tools_node(state: AgentState) -> AgentState:
    """
    Execute all tool calls from the last AI message.
    Returns ToolMessage results that get appended to conversation history.
    """
    last_message = state["messages"][-1]

    tool_results = []

    for tool_call in last_message.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]

        if tool_name in TOOL_MAP:
            try:
                result = TOOL_MAP[tool_name].invoke(tool_args)
            except Exception as e:
                result = f"Error invoking tool {tool_name}: {e}"

        else:
            result = f"Tool {tool_name} not found."

        tool_results.append(
            ToolMessage(
                content=str(result),
                tool_call_id=tool_call["id"],
                name=tool_name,
            )
        )        

    return {
        "messages": tool_results,
    }

In [6]:
# ── Routing Logic ─────────────────────────────────────────────────────────────
def should_continue(state: AgentState) -> str:
    """
    Conditional edge function — determines next node after model_node.

    Returns:
      "tools"  → LLM made tool calls, execute them
      "end"    → LLM gave final answer OR iteration limit reached
    """
    last_message = state["messages"][-1]

    if state["iteration_count"] >= state["max_iterations"]:
        print(f"  [Graph] Iteration limit ({state['max_iterations']}) reached — stopping")
        return "end"
    
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        tool_names = [tc["name"] for tc in last_message.tool_calls]
        print(f"  [Graph] Model called tools: {tool_names} — executing them")
        return "tools"
    print(f"  [Graph] Final answer reached after {state['iteration_count']} iterations")
    return "end"

In [8]:
# ── Build Graph ───────────────────────────────────────────────────────────────
from langgraph import graph


def build_graph() -> StateGraph:
    """
    Structure:
    START → model_node → [conditional] → tools_node → model_node (loop) → END
    """

    graph = StateGraph(AgentState)

    graph.add_node("model", model_node)
    graph.add_node("tools", tools_node)

    graph.set_entry_point("model")

    graph.add_conditional_edges(
        "model",
        should_continue,
        {
            "tools": "tools",
            "end": END,
        },
    )

    graph.add_edge("tools", "model")

    return graph

checkpointer = MemorySaver()

compiled_graph = build_graph().compile(checkpointer=checkpointer)

In [9]:
# ── Compile and Run ───────────────────────────────────────────────────────────

def run_agent(question: str, thread_id: str = "default") -> str:
    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print(f"{'='*60}")

    config = {"configurable": {"thread_id": thread_id}}

    initial_state = {
        "messages": [HumanMessage(content=question)],
        "iteration_count": 0,
        "max_iterations": 5,
    }

    result = compiled_graph.invoke(initial_state, config=config)

    final_message = result["messages"][-1]
    answer = final_message.content if isinstance(final_message, AIMessage) else "No answer generated."

    tool_messages = [msg for msg in result["messages"] if isinstance(msg, ToolMessage)]
    if tool_messages:
        print("\nTool Calls Made:")
        for tm in tool_messages:
            print(f"{tm.name}")
    
    print(f"Answer: {answer}")
    return answer

In [10]:
def main():
    print("LANGGRAPH AGENT — BUILT FROM SCRATCH")
    print("Explicit nodes, edges, state, and conditional routing\n")


    questions = [
        "What is 15 multiplied by 23, and what is that result squared?",
        "What should I know about JWT tokens for my backend API?",
        "Get the fact about PostgreSQL, then calculate 2 to the power of 10.",
        "What is the capital of Mars?",  # no tool needed — direct LLM answer
    ]

    for i, q in enumerate(questions):
        run_agent(q, thread_id=f"session_{i}")  # use part of question as thread ID for memory grouping


    print(f"\n{'='*60}")
    print("MULTI-TURN CONVERSATION (same thread_id)")
    print("="*60)

    run_agent("Tell me about Redis caching.", thread_id="persistent")
    run_agent("And how does that compare to in-memory Python dicts?", thread_id="persistent")


if __name__ == "__main__":
    main()

LANGGRAPH AGENT — BUILT FROM SCRATCH
Explicit nodes, edges, state, and conditional routing


Question: What is 15 multiplied by 23, and what is that result squared?
  [Graph] Final answer reached after 1 iterations
Answer: - 15 × 23 = 345
- 345^2 = 119,025

So the product is 345, and its square is 119,025.

Question: What should I know about JWT tokens for my backend API?
  [Graph] Final answer reached after 1 iterations
Answer: Here’s a concise, practical guide to JWTs for a backend API, plus what you should watch out for.

What JWTs are (in short)
- JWT stands for JSON Web Token. It’s a compact token that carries claims (information) about a user or client.
- It has three parts: header, payload, signature. The payload is base64-encoded and readable (not encrypted), so don’t put secrets or PII there.
- You typically use JWTs to prove authentication/authorization without server-side sessions (stateless auth).

What to put in a JWT (typical claims)
- iss (issuer): who issued the token
-